# Baseline Tool-Call Evaluation

Evaluate the untrained Hugging Face base model directly on the held-out tool-call cases. This uses the same local generation path as the SFT evaluation notebook, but without loading a LoRA adapter.

In [1]:
from pathlib import Path
import csv
import json
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from toolcall_rl.evaluation.cases import SEED_EVAL_CASES
from toolcall_rl.evaluation.schemas import SYSTEM_PROMPT
from toolcall_rl.evaluation.scoring import score_response

MODEL_ID = "HuggingFaceTB/SmolLM-1.7B-Instruct"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

len(SEED_EVAL_CASES), MODEL_ID

(20, 'HuggingFaceTB/SmolLM-1.7B-Instruct')

## Load Base Model

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype)
model.to(device)
model.eval()

device

/home/shubeeksh/projects/toolcall-rl/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 218/218 [00:00<00:00, 237.09it/s]


'cuda'

## Generation Helpers

In [3]:
def render_messages(messages):
    if tokenizer.chat_template:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    lines = []
    for message in messages:
        lines.append(f"<|{message['role']}|>\n{message['content']}")
    lines.append("<|assistant|>\n")
    return "\n".join(lines)


def generate_response(prompt, max_new_tokens=160):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    rendered = render_messages(messages)
    inputs = tokenizer(rendered, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


generate_response("What is 24 * 17?", max_new_tokens=80)

'24 * 17 is a mathematical expression that can be simplified to 363.'

## Run Baseline Evaluation

In [4]:
from dataclasses import asdict

results = []
for case in SEED_EVAL_CASES:
    response = generate_response(case.prompt)
    score = score_response(response, case)
    results.append(
        {
            "case": asdict(case),
            "response": response,
            "score": asdict(score),
        }
    )

len(results)

20

## Build Table

In [5]:
def flatten_result(index, result):
    case = result["case"]
    score = result["score"]
    return {
        "case": index,
        "prompt": case["prompt"],
        "expected_tool": case["expected_tool"],
        "expected_args": json.dumps(case["expected_args"], ensure_ascii=False),
        "response": result["response"],
        "valid_json": score["valid_json"],
        "json_only": score["json_only"],
        "tool_match": score["tool_match"],
        "args_match": score["args_match"],
        "total_reward": score["total_reward"],
    }


table_rows = [flatten_result(index, result) for index, result in enumerate(results, start=1)]

## Summary

In [6]:
total_cases = len(table_rows)
max_reward = total_cases * 4
total_reward = sum(row["total_reward"] for row in table_rows)
summary = {
    "model": MODEL_ID,
    "cases": total_cases,
    "valid_json": sum(row["valid_json"] for row in table_rows),
    "json_only": sum(row["json_only"] for row in table_rows),
    "tool_match": sum(row["tool_match"] for row in table_rows),
    "args_match": sum(row["args_match"] for row in table_rows),
    "total_reward": total_reward,
    "max_reward": max_reward,
}
summary

{'model': 'HuggingFaceTB/SmolLM-1.7B-Instruct',
 'cases': 20,
 'valid_json': 2,
 'json_only': 0,
 'tool_match': 0,
 'args_match': 0,
 'total_reward': 2,
 'max_reward': 80}

## Results Table

In [7]:
import pandas as pd

display(pd.DataFrame(table_rows))

,case,prompt,expected_tool,expected_args,response,valid_json,json_only,tool_match,args_match,total_reward
0,1,Work out (73 * 9) - 14.,calculator,"{""expression"": ""(73 * 9) - 14""}",Workout (73 * 9) - 14.\n\nThe answer is:,0,0,0,0,0
1,2,Search Google for current LoRA adapter merging...,google_search,"{""query"": ""current LoRA adapter merging tutori...",Here are some current LoRA adapter merging tut...,0,0,0,0,0
2,3,I have 27.5 miles; express that in kilometers.,unit_converter,"{""value"": 27.5, ""from_unit"": ""miles"", ""to_unit...","To convert miles to kilometers, you need to mu...",0,0,0,0,0
3,4,"Give text statistics for: ""Adapters are compac...",text_stats,"{""text"": ""Adapters are compact. Rewards improv...",Here are the text statistics for the given sen...,0,0,0,0,0
4,5,"Convert ""REWARD SIGNAL"" to lowercase.",string_formatter,"{""text"": ""REWARD SIGNAL"", ""operation"": ""lowerc...","The task is to convert ""REWARD SIGNAL"" to lowe...",0,0,0,0,0
5,6,Get the weather for Madrid using celsius units.,weather_lookup,"{""city"": ""Madrid"", ""unit"": ""celsius""}",Here is the implementation of the function usi...,0,0,0,0,0
6,7,Exchange 325 USD into CAD.,currency_converter,"{""amount"": 325, ""from_currency"": ""USD"", ""to_cu...",Here is the code to exchange 325 USD into CAD:...,0,0,0,0,0
7,8,"Translate ""machine learning"" from English into...",translate_text,"{""text"": ""machine learning"", ""source_language""...",O que é machine learning?,0,0,0,0,0
8,9,"Add ""Evaluation review"" to my calendar on 2026...",create_calendar_event,"{""title"": ""Evaluation review"", ""date"": ""2026-0...","Here is an example of how you could add ""Evalu...",1,0,0,0,1
9,10,"Email qa@example.com with subject ""Test result...",send_email,"{""recipient"": ""qa@example.com"", ""subject"": ""Te...",Subject: Test result\n\nMessage: All held-out ...,0,0,0,0,0


## Save Results

In [8]:
jsonl_path = OUTPUT_DIR / "baseline_eval_results.jsonl"
csv_path = OUTPUT_DIR / "baseline_eval_results.csv"

with jsonl_path.open("w", encoding="utf-8") as file:
    for result in results:
        file.write(json.dumps(result, ensure_ascii=False) + "\n")

with csv_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=table_rows[0].keys())
    writer.writeheader()
    writer.writerows(table_rows)

jsonl_path, csv_path

(PosixPath('/home/shubeeksh/projects/toolcall-rl/outputs/baseline_eval_results.jsonl'),
 PosixPath('/home/shubeeksh/projects/toolcall-rl/outputs/baseline_eval_results.csv'))